# Estatísticas dos datasets (F2, entrega 2.5)

Gera as tabelas da seção 6 do relatório para os datasets que estiverem em `data/`:

1. **Resumo** de cada dataset (fluxos, hosts e proporção de ataques), ao lado da Tabela 1 do artigo.
2. **Contagem por split**: fluxos de cada categoria de ataque no treino, na validação e no teste, e os hosts de cada split.

As tabelas são salvas em CSV e LaTeX em `results/estatisticas/`. A lógica está em `graphids/data/stats.py`; este notebook só executa e mostra os resultados.

Datasets ausentes em `data/` são pulados. Os grafos vêm do cache (`data/cache/`), então a segunda execução é rápida.

In [ ]:
import os
from pathlib import Path

import pandas as pd
from IPython.display import display

from graphids.config import load_config
from graphids.data.loading import csv_path
from graphids.data.stats import main as gerar_estatisticas

# O notebook roda a partir de notebooks/; os caminhos do projeto são relativos à raiz.
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)


DATA_DIR = "data"
SEED = 1
OUT = Path("results/estatisticas")

## Quais datasets estão disponíveis

Cada config de `configs/` define o dataset, a `fraction` e se os timestamps são descartados. A seed usada é a mesma das rodadas de reprodução

In [ ]:
configs = []
for path in sorted(Path("configs").glob("NF-*.yaml")):
    dataset = load_config(path).dataset
    try:
        csv_path(DATA_DIR, dataset)
        configs.append(str(path))
        print(f"ok      {dataset}")
    except FileNotFoundError:
        print(f"ausente {dataset}")

## Gerar as tabelas

In [ ]:
args = [a for c in configs for a in ("--config", c)]
resumo = gerar_estatisticas(
    args + ["--data-dir", DATA_DIR, "--seed", str(SEED), "--out", str(OUT)]
)

## Resumo x Tabela 1 do artigo

In [ ]:
display(resumo)

## Contagem por categoria em cada split

In [ ]:
for csv in sorted(OUT.glob("*_contagem_por_split.csv")):
    print(csv.stem.replace("_contagem_por_split", ""))
    display(pd.read_csv(csv, index_col=0))